# Evaluating agents with Inspect AI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meta-pytorch/OpenEnv/blob/main/examples/evaluation_inspect.ipynb)

After training a model in an OpenEnv environment, you need to measure how it actually performs on a held-out set of episodes. This notebook shows how to use [Inspect AI](https://inspect.aisi.org.uk/) through OpenEnv's `InspectAIHarness` to run structured, reproducible evals.

**What you'll build:**
- An Inspect AI `Task` whose solver calls an OpenEnv environment
- A custom scorer that grades the env's response
- A reproducible eval run tracked as an `EvalResult`

**Model providers supported:** OpenAI, Anthropic, or any open model via Hugging Face Inference Providers.

---

## 1. Install dependencies

In [ ]:
!pip install -q "inspect-ai>=0.3.0"
!pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"

---

## 2. Set your model provider

Uncomment exactly one option. All three options feed into the same task and harness in
Section 4 — no other cells need to change.

- **Options A/B** (OpenAI / Anthropic): Inspect AI's `generate()` calls the model.
- **Option C** (HF open model): the solver calls `InferenceClient` directly and bypasses
  `generate()`, so no OpenAI/Anthropic key is needed.

In [ ]:
import os
from huggingface_hub import InferenceClient

hf_client = None  # overridden by Option C

# --- Option A: OpenAI ---
os.environ["OPENAI_API_KEY"] = "sk-..."  # replace with your key
MODEL = "openai/gpt-4o-mini"

# --- Option B: Anthropic ---
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# MODEL = "anthropic/claude-haiku-4-5-20251001"

# --- Option C: open model via HF Inference Providers ---
# The solver calls InferenceClient directly, so generate() is never invoked.
# MODEL is passed to InspectAIHarness for logging only.
# try:
#     from google.colab import userdata
#     os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
# except Exception:
#     pass
# HF_MODEL_ID = "meta-llama/Llama-3.3-70B-Instruct"
# hf_client = InferenceClient(
#     model=HF_MODEL_ID,
#     token=os.environ.get("HF_TOKEN", ""),
# )
# MODEL = f"hf/{HF_MODEL_ID}"  # stored in EvalConfig for reproducibility

---

## 3. How Inspect AI and OpenEnv fit together

- **OpenEnv** provides the environment: `reset()`, tool calls, reward.
- **Inspect AI** provides the eval infrastructure: datasets, solvers, scorers, and structured logs.
- **`InspectAIHarness`** is the bridge — it wraps `inspect_ai.eval()` so runs are tracked
  with `EvalConfig` / `EvalResult`.

An Inspect AI `Task` has three parts:
1. **Dataset** — samples with `input` and `target` fields
2. **Solver** — calls the model and the env for each sample
3. **Scorer** — grades the result

> **Note on the env client:** `echo_env` is a pure MCP environment. Interact with it via
> `MCPToolClient` and `call_tool("echo_message", ...)`. For non-MCP environments, use
> `GenericEnvClient` instead.

---

## 4. Define the task

The solver handles both model paths transparently:
- If `hf_client` is set (Option C), it calls `InferenceClient` in a thread executor.
- Otherwise (Options A/B), it delegates to Inspect AI's `generate()`.

Everything else — the dataset, scorer, and harness — is identical across all providers.

In [ ]:
import asyncio

from inspect_ai import Task, task
from inspect_ai.dataset import Sample
from inspect_ai.scorer import CORRECT, INCORRECT, Score, Target, accuracy, scorer
from inspect_ai.solver import Generate, TaskState, solver

from openenv.core import MCPToolClient

ECHO_ENV_URL = "https://openenv-echo-env.hf.space"


@task
def openenv_echo_eval(base_url: str = ECHO_ENV_URL):
    """Evaluate whether a model can repeat a phrase exactly via echo_env."""
    return Task(
        dataset=[
            Sample(input="Repeat exactly: hello world", target="hello world"),
            Sample(input="Repeat exactly: inspect ai", target="inspect ai"),
            Sample(input="Repeat exactly: openenv eval", target="openenv eval"),
            Sample(input="Repeat exactly: reinforcement learning", target="reinforcement learning"),
            Sample(input="Repeat exactly: hugging face", target="hugging face"),
        ],
        solver=echo_env_solver(base_url=base_url, client=hf_client),
        scorer=echo_scorer(),
    )


@solver
def echo_env_solver(base_url: str, client=None):
    """Unified solver: uses InferenceClient if provided, else Inspect AI's generate()."""

    async def solve(state: TaskState, generate: Generate) -> TaskState:
        if client is not None:
            # HF path: InferenceClient is synchronous — run it in a thread
            loop = asyncio.get_running_loop()
            resp = await loop.run_in_executor(
                None,
                lambda: client.chat_completion(
                    messages=[{"role": "user", "content": state.user_prompt.text}],
                    max_tokens=50,
                ),
            )
            model_output = resp.choices[0].message.content.strip()
        else:
            # OpenAI / Anthropic path: let Inspect AI call the model
            state = await generate(state)
            model_output = state.output.completion.strip()

        env = MCPToolClient(base_url=base_url)
        try:
            await env.reset()
            echoed = await env.call_tool("echo_message", message=model_output)
            state.metadata["echoed"] = str(echoed) if echoed is not None else ""
        finally:
            await env.close()

        return state

    return solve


@scorer(metrics=[accuracy()])
def echo_scorer():
    """CORRECT if the env echoed back exactly what the target phrase was."""

    async def score(state: TaskState, target: Target) -> Score:
        echoed = state.metadata.get("echoed", "").strip()
        expected = target.text.strip()
        return Score(
            value=CORRECT if echoed == expected else INCORRECT,
            explanation=f"Env echoed {echoed!r}, expected {expected!r}",
        )

    return score

---

## 5. Run the eval with `InspectAIHarness`

`EvalConfig` captures everything needed to reproduce the run — including which model was used.
For Option C, `MODEL` stores the HF model ID so the config remains reproducible even though
`generate()` is bypassed.

In [ ]:
import inspect_ai
import openenv

from openenv.core.evals import EvalConfig, EvalResult, InspectAIHarness

harness = InspectAIHarness(log_dir="./eval-logs")

config = EvalConfig(
    harness_name="InspectAIHarness",
    harness_version=inspect_ai.__version__,
    library_versions={"openenv": openenv.__version__},
    dataset="openenv_echo_eval",
    eval_parameters={
        "model": MODEL,
        "task": openenv_echo_eval(base_url=ECHO_ENV_URL),
        "temperature": 0.0,
    },
)

result: EvalResult = harness.run_from_config(config)
print("Scores:", result.scores)

---

## 6. Inspect the results

`EvalResult` carries both the config and the scores, making it easy to log, compare across
runs, or push to the Hub.

In [ ]:
import json

print(json.dumps(result.model_dump(), indent=2))

In [ ]:
# Inspect AI also writes detailed logs to ./eval-logs/
# !inspect view --log-dir ./eval-logs

---

## 7. Adapt to your own environment

1. **Dataset** — held-out episodes from your env. Each `Sample` needs `input` and `target`.
2. **Solver** — same pattern: `client is not None` for HF, else `generate()` for proprietary APIs.
   Call your env's MCP tool in place of `echo_message`.
3. **Scorer** — compare the tool result against the target, or use the env's reward signal.

In [ ]:
# Template: plug in your own MCP environment
from inspect_ai.solver import solver, TaskState, Generate
from openenv.core import MCPToolClient


@solver
def my_env_solver(base_url: str, client=None):
    async def solve(state: TaskState, generate: Generate) -> TaskState:
        if client is not None:
            loop = asyncio.get_running_loop()
            resp = await loop.run_in_executor(
                None,
                lambda: client.chat_completion(
                    messages=[{"role": "user", "content": state.user_prompt.text}],
                    max_tokens=200,
                ),
            )
            model_output = resp.choices[0].message.content.strip()
        else:
            state = await generate(state)
            model_output = state.output.completion.strip()

        env = MCPToolClient(base_url=base_url)
        try:
            await env.reset()
            result = await env.call_tool("your_tool_name", message=model_output)
            state.metadata["env_result"] = result
        finally:
            await env.close()
        return state

    return solve

---

## Next steps

- **[Wordle GRPO tutorial](https://github.com/meta-pytorch/OpenEnv/blob/main/docs/source/tutorials/wordle-grpo.md)** — train a model you can then evaluate with this notebook
- **[Rubrics tutorial](https://github.com/meta-pytorch/OpenEnv/blob/main/docs/source/tutorials/rubrics.md)** — define composable reward functions inside the environment
- **[Inspect AI documentation](https://inspect.aisi.org.uk/)** — full reference for tasks, solvers, scorers, and the log viewer